# VascuQuest JAX Backend — One-Subject Qualification v2

This notebook qualifies the repaired bounded-replay JAX Virtual Disease solver in PR #20. The integration equations and adaptive SSP-RK2 path remain unchanged; only the artificial fixed 500,000-step/history-memory coupling has been removed.

Exactly one canonical PWDB subject is exercised across all four frozen disease conditions. A PASS is software/mechanistic backend qualification only, not clinical validation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = 'https://github.com/KNOWDYN/VascuQuest.git'
QUALIFICATION_REF = 'release/parameterized-cohort-qualification'
LOCAL_REPO = Path('/content/VascuQuest-jax-qualification-v2')
LOCAL_SOURCE = Path('/content/vascuquest-pwdb-source')
LOCAL_XDG = Path('/content/vascuquest-xdg-v2')
OUTPUT_BASE = Path('/content/drive/MyDrive/VascuQuest/jax_one_subject_qualification_v2')

drive_candidates = [
    Path('/content/drive/MyDrive/VQ_WallWork_CBM/source/PWDB_3275625'),
    Path('/content/drive/Shareddrives/VQ_WallWork_CBM/source/PWDB_3275625'),
]
DRIVE_SOURCE = next((p for p in drive_candidates if p.exists()), drive_candidates[0])

for path in (LOCAL_SOURCE, LOCAL_XDG, OUTPUT_BASE):
    path.mkdir(parents=True, exist_ok=True)
os.environ['XDG_DATA_HOME'] = str(LOCAL_XDG / 'data')
os.environ['XDG_CACHE_HOME'] = str(LOCAL_XDG / 'cache')
os.environ['XDG_STATE_HOME'] = str(LOCAL_XDG / 'state')

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', QUALIFICATION_REF, REPO_URL, str(LOCAL_REPO)], check=True)
CODE_REVISION = subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT_ROOT = OUTPUT_BASE / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(LOCAL_REPO) + '[jax]'], check=True)
import jax
gpu_devices = [d for d in jax.devices() if getattr(d, 'platform', '') == 'gpu']
print('Code revision:', CODE_REVISION)
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
if not gpu_devices:
    raise RuntimeError('No JAX GPU device detected. In Colab select Runtime > Change runtime type > T4 GPU, then restart and rerun from this cell.')
print('JAX GPU gate: PASS ->', gpu_devices)
print('Configured PWDB Drive source:', DRIVE_SOURCE)
print('Revision-scoped output:', OUTPUT_ROOT)


## Stage canonical PWDB artifacts to local SSD

No recursive Drive search is performed. Existing exact files are copied once; missing canonical artifacts are checksum-verified through VascuQuest acquisition.

In [ ]:
STAGER = LOCAL_REPO / 'tests/full_data/parameterized_cohort_colab_stage.py'
STAGE_REPORT = OUTPUT_ROOT / 'source_stage.json'
stage = subprocess.run([sys.executable, str(STAGER), '--drive-source-dir', str(DRIVE_SOURCE), '--local-source', str(LOCAL_SOURCE), '--report', str(STAGE_REPORT)])
if stage.returncode != 0:
    raise RuntimeError(f'PWDB staging failed with exit code {stage.returncode}')
print(STAGE_REPORT.read_text())
print('PWDB local-SSD source gate: PASS')


## Run repaired one-subject JAX qualification

The same NumPy↔JAX operator gates are retained. Full JAX solves now use a subject-specific adaptive safety cap and bounded-memory source-grid replay.

In [ ]:
RUNNER = LOCAL_REPO / 'tests/full_data/jax_one_subject_qualification_v2.py'
REPORT = OUTPUT_ROOT / 'jax-one-subject-qualification.json'
completed = subprocess.run([sys.executable, str(RUNNER), '--source', str(LOCAL_SOURCE), '--report', str(REPORT), '--code-revision', CODE_REVISION])
if completed.returncode != 0:
    if REPORT.exists():
        print('Persisted failure record:')
        print(REPORT.read_text())
    raise RuntimeError(f'JAX qualification failed with exit code {completed.returncode}')


## Qualification summary

In [ ]:
report = json.loads(REPORT.read_text())
print(json.dumps({
    'status': report.get('status'),
    'code_revision': report.get('code_revision'),
    'canonical_subject_id': report.get('canonical_subject_id'),
    'source_age_years': report.get('source_age_years'),
    'elapsed_seconds': report.get('elapsed_seconds'),
    'anchor': report.get('full_numpy_jax_anchor'),
}, indent=2, sort_keys=True))
for case in report.get('cases', []):
    timing = case['jax_full_solve'].get('timing', {})
    print(
        case['condition'],
        'operator=', case['operator_equivalence']['passed'],
        'converged=', case['jax_full_solve']['diagnostics']['converged'],
        'wall_s=', round(case['jax_full_solve']['wall_seconds'], 3),
        'steps=', timing.get('final_cycle_steps'),
        'cap=', timing.get('max_steps_per_cycle'),
        'output_samples=', timing.get('output_samples'),
        'device=', timing.get('device'),
    )
if report.get('status') != 'PASS':
    raise RuntimeError('PR #20 repaired JAX qualification is not PASS')
print('PR #20 repaired JAX one-subject qualification: PASS')
print('Durable report:', REPORT)


Do not merge PR #20 solely from the notebook banner. Inspect the durable JSON for operator equivalence, convergence, final-cycle step counts, device, timing, and the NumPy/JAX anchor before merge.